### Kaggle version — Llama 3.2 11B Vision CoT Generation

This notebook has been adapted from the original Google Colab version to run on **Kaggle Notebooks**.

Before running:
1. In the notebook Settings panel (right sidebar), turn **Internet** ON (Settings → Internet → On).
2. Set **Accelerator** to a GPU (T4 x2 or P100 both work; a single T4 is enough for 4-bit inference/LoRA).
3. Add your dataset zip (e.g. `SLAKE_split_en_realimages.zip`) via **Add Data → Upload** or as a Kaggle Dataset — it will then be available under `/kaggle/input/<dataset-name>/...`.
4. If `unsloth/Llama-3.2-11B-Vision-Instruct` requires accepting a license on Hugging Face, add your HF token as a Kaggle **Secret** named `HF_TOKEN` (Add-ons → Secrets) so the login cell below can use it.

All Colab-specific code (`google.colab.drive`, `COLAB_` env checks, `/content/...` paths) has been replaced with Kaggle-native equivalents (`/kaggle/input`, `/kaggle/working`, Kaggle Secrets).


### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [2]:
%%capture
import os, re

# Kaggle already ships recent torch/cuda, but we still need unsloth + friends.
# (Colab's COLAB_ env-var branch has been replaced with a KAGGLE_ check.)
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install -q --no-deps --upgrade "torchao>=0.16.0"
else:
    # local / other cloud setups
    !pip install -q unsloth

!pip install -q transformers==4.56.2
!pip install -q --no-deps trl==0.22.2


In [3]:
# Optional: log in to Hugging Face if the base model is gated.
# Store your token as a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets),
# then this cell will pick it up automatically. Safe to skip if not needed.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
except Exception as e:
    print(f"Skipping HF login (no secret configured or not needed): {e}")


Skipping HF login (no secret configured or not needed): Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 130489018 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.


### Unsloth

In [4]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit", # Llama 3.2 vision support
    "unsloth/Llama-3.2-11B-Vision-bnb-4bit",
    "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit", # Can fit in a 80GB card!
    "unsloth/Llama-3.2-90B-Vision-bnb-4bit",

    "unsloth/Pixtral-12B-2409-bnb-4bit",              # Pixtral fits in 16GB!
    "unsloth/Pixtral-12B-Base-2409-bnb-4bit",         # Pixtral base model

    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",          # Qwen2 VL support
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit",

    "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",      # Any Llava variant works!
    "unsloth/llava-1.5-7b-hf-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Llama-3.2-11B-Vision-Instruct",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.16: Fast Mllama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [5]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

<a name="Data"></a>
### Data Prep


In [6]:
# Kaggle doesn't use Google Drive. Datasets you attach via "Add Data" are
# mounted read-only under /kaggle/input/<dataset-slug>/...
# List what's available so you can confirm the correct path below.
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))


/kaggle/input/datasets/arups330/slackdataset/split_summary.csv
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/val/xmlab418_source.jpg
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/val/open.csv
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/val/closed.csv
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/val/xmlab404_source.jpg
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/val/xmlab408_source.jpg
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/test/open.csv
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/test/closed.csv
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/test/xmlab405_source.jpg
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/test/xmlab407_source.jpg
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/train/xmlab411_source.jpg
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/CT/train/xmlab414_source.jpg
/kaggle/input/datasets/arups330/slackdataset/Brain_Face/

Let's take a look at the dataset, and check what the 1st example shows:

In [7]:
# Point this at the zip file as it appears under /kaggle/input (see the
# listing printed above). Update the dataset-slug/filename to match yours.
#ZIP_PATH_IN_DRIVE = "/kaggle/input/slake-split-en-realimages/SLAKE_split_en_realimages.zip"


In [8]:
# import zipfile
# import os

# EXTRACT_DIR = "/kaggle/working/abdomen_dataset"

# if not os.path.exists(EXTRACT_DIR):
#     print("Extracting from Drive (first time this session)...")
#     os.makedirs(EXTRACT_DIR, exist_ok=True)
#     with zipfile.ZipFile(ZIP_PATH_IN_DRIVE, "r") as zf:
#         zf.extractall(EXTRACT_DIR)
#     print(f"Extracted to {EXTRACT_DIR}")
# else:
#     print(f"Already extracted at {EXTRACT_DIR}, skipping.")


In [9]:

# ---------------------------------------------------------------------------
# 1d. Auto-locate the "Abdomen" folder inside the extracted contents
#     (handles the extra wrapping folder you mentioned).
# ---------------------------------------------------------------------------
# def find_abdomen_folder(base: str) -> str:
#     for root, dirs, files in os.walk(base):
#         for d in dirs:
#             if d.lower() == "abdomen":
#                 return os.path.join(root, d)
#     raise FileNotFoundError(
#         f"Could not find a folder named 'Abdomen' anywhere under {base}. "
#         f"Top-level contents were: {os.listdir(base)}"
#     )

ABDOMEN_ROOT = "/kaggle/input/datasets/arups330/slack-abdomen-dataset/Abdomen"
NECK_ROOT  = "/kaggle/input/datasets/arups330/slackdataset/Neck"
import os
for root, dirs, files in os.walk("/kaggle/input"):
    if root.endswith("Neck"):
        print(root)


/kaggle/input/datasets/arups330/slackdataset/Neck


In [10]:
# ---------------------------------------------------------------------------
# 4. Load your Abdomen dataset -- walks BOTH modality folders (CT, MRI, or
#    however yours are named) automatically, using only the "train" split.
# ---------------------------------------------------------------------------
import os
import glob
import pandas as pd
from PIL import Image

train_csvs = glob.glob(os.path.join(NECK_ROOT, "*", "train", "*.csv"))
print(f"Found {len(train_csvs)} training CSVs:")
for p in train_csvs:
    print(" ", p)

if not train_csvs:
    raise FileNotFoundError(
        f"No CSVs found under {NECK_ROOT}/*/train/*.csv -- check ABDOMEN_ROOT "
        f"and confirm your folder structure matches Neck/<CT or MRI>/train/*.csv"
    )

train_frames = []
for csv_path in train_csvs:
    split_dir = os.path.dirname(csv_path)   # e.g. Abdomen/CT/train
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    train_frames.append(df)

train_df = pd.concat(train_frames, ignore_index=True)
print(f"\nTotal training rows: {len(train_df)}")
print("Columns:", list(train_df.columns))

# Handle either column name -- "img_name" (original SLAKE export) or
# "image_file" (the real-image-files export from earlier).
IMG_COL = "image_file" if "image_file" in train_df.columns else "img_name"
print(f"Using image column: {IMG_COL}")

# Some img_name values may look like "xmlab1/source.jpg" (nested path from
# the original SLAKE export) even though the file itself sits flat in the
# split folder -- this helper checks both possibilities.
def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(img_name)  # handles "xmlab1/source.jpg" -> "source.jpg"
    candidates = [
        os.path.join(split_dir, img_name),
        os.path.join(split_dir, flat_name),
        os.path.join(split_dir, img_name.replace("/", "_")),  # matches earlier flatten step
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")



Found 2 training CSVs:
  /kaggle/input/datasets/arups330/slackdataset/Neck/CT/train/open.csv
  /kaggle/input/datasets/arups330/slackdataset/Neck/CT/train/closed.csv

Total training rows: 129
Columns: ['image_file', 'img_id', 'location', 'modality', 'question', 'answer', 'q_lang', 'answer_type', 'content_type', 'base_type', 'qid', 'triple', 'split_dir']
Using image column: image_file


In [11]:

# ---------------------------------------------------------------------------
# 6. Your exact CoT instruction template
# ---------------------------------------------------------------------------
INSTRUCTION_TEMPLATE = """Context: You are a senior radiologist performing structured diagnostic reasoning.

Your goal is NOT just to answer, but to produce a **step-by-step clinical reasoning chain (Chain-of-Thought)** grounded in the image.

-----------------------------------
INPUT:
- Question: {question}
- Ground Truth Answer: {answer}
-----------------------------------

TASK INSTRUCTIONS:

You MUST follow a strict multi-step reasoning process:

Step 1: Identify Imaging Modality
- Determine modality (X-ray / CT / MRI / Ultrasound)
- Explain visual clues (contrast, density, grayscale pattern)

Step 2: Global Image Understanding
- Describe anatomical region
- Identify orientation (axial, sagittal, coronal, frontal)

Step 3: Region-wise Analysis
- Divide image into anatomical zones
- Analyze each region systematically

Step 4: Visual Feature Extraction
- Density (hyperdense / hypodense)
- Shape, edges, symmetry
- Texture abnormalities

Step 5: Abnormality Detection
- Identify pathology (if present)
- Localize precisely

Step 6: Clinical Reasoning
- Link findings to medical knowledge
- Explain WHY the abnormality matches the condition

Step 7: Question Understanding
- What exactly is the question asking?
- Type: (yes/no, location, modality, abnormality)

Step 8: Answer Justification
- Justify the provided answer: "{answer}"
- Explain why it is correct based on image evidence

-----------------------------------
OUTPUT FORMAT (STRICT):

1. Imaging Modality:
2. Anatomical Region:
3. Orientation:
4. Region-wise Findings:
5. Key Visual Features:
6. Detected Abnormality:
7. Clinical Interpretation:
8. Question Analysis:
9. Final Answer Justification:

IMPORTANT:
- Do NOT skip steps
- Do NOT give short answers
- Each step must contain 2-4 sentences
- Use medical terminology
"""

def generate_cot(image: Image.Image, question: str, answer: str) -> str:
    prompt_text = INSTRUCTION_TEMPLATE.format(question=question, answer=answer)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_text},
                {"type": "image", "image": image},
            ],
        }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

    output_ids = model.generate(
        **inputs,
        max_new_tokens=1024,   # CoT is long: 9 sections x 2-4 sentences each
        use_cache=True,
        temperature=0.3,
        min_p=0.1,
    )
    return tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()



In [12]:

# 7. Generate CoT for every row in train_df (in place, as a new column)
#    NOTE: This only touches train_df in memory + writes new output files.
#    Your original Drive zip / extracted Abdomen folder are never modified.
# ---------------------------------------------------------------------------
print(f"Generating CoT for {len(train_df)} training rows...")

cots = []
for i, row in train_df.iterrows():
    try:
        img_path = resolve_image_path(row["split_dir"], row[IMG_COL])
        image = Image.open(img_path).convert("RGB")
        cot = generate_cot(image, row["question"], row["answer"])
    except Exception as e:
        print(f"  [WARN] row {i} failed ({row.get(IMG_COL)}): {e}")
        cot = ""
    cots.append(cot)

    if i % 10 == 0:
        print(f"  [{i}/{len(train_df)}] done")

train_df["CoT"] = cots
print("\nCoT generation complete.")


Generating CoT for 129 training rows...
  [0/129] done
  [10/129] done
  [20/129] done
  [30/129] done
  [40/129] done
  [50/129] done
  [60/129] done
  [70/129] done
  [80/129] done
  [90/129] done
  [100/129] done
  [110/129] done
  [120/129] done

CoT generation complete.


In [19]:
train_df.head()

,image_file,img_id,location,modality,question,answer,q_lang,answer_type,content_type,base_type,qid,triple,split_dir,CoT
0,xmlab410_source.jpg,410,Neck,CT,How was this image taken?,CT,en,OPEN,Modality,vqa,3005,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/N...,**Step 1: Imaging Modality**\n\nThe image prov...
1,xmlab410_source.jpg,410,Neck,CT,What is the scanning plane of this image?,Transverse Plane,en,OPEN,Plane,vqa,3006,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/N...,**Step 1: Imaging Modality**\n\nThe image is a...
2,xmlab410_source.jpg,410,Neck,CT,Where does the image represent in the body?,Neck,en,OPEN,Position,vqa,3007,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/N...,**Step 1: Imaging Modality**\n\nThe provided i...
3,xmlab410_source.jpg,410,Neck,CT,Which is larynx in this image?,Black Hollow,en,OPEN,Position,vqa,3008,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/N...,**Step 1: Imaging Modality**\n\nThe provided i...
4,xmlab410_source.jpg,410,Neck,CT,Which organ system is imaged?,Neck,en,OPEN,Organ,vqa,3009,['vhead' '_' '_'],/kaggle/input/datasets/arups330/slackdataset/N...,**Step 1: Imaging Modality**\n\nThe provided i...


In [24]:
import base64
from IPython.display import HTML

csv_path = "/kaggle/working/train_df_with_cot.csv"
train_df.to_csv(csv_path, index=False)

with open(csv_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

html = f'<a download="train_df_with_cot.csv" href="data:text/csv;base64,{b64}" target="_blank">⬇️ Click here to download train_df_with_cot.csv</a>'
HTML(html)

In [21]:
import os

for root, dirs, files in os.walk(OUTPUT_ROOT):
    for f in files:
        print(os.path.join(root, f))

/kaggle/working/Neck_train_with_CoT/CT/train/open.csv
/kaggle/working/Neck_train_with_CoT/CT/train/closed.csv


In [22]:
# ---------------------------------------------------------------------------
# 8. Save back out to per-CSV files (grouped by original split_dir), into a
#    SEPARATE local output folder -- never overwriting your original files.
# ---------------------------------------------------------------------------
OUTPUT_ROOT = "/kaggle/working/Neck_train_with_CoT"

for split_dir, group_df in train_df.groupby("split_dir"):
    rel_dir = os.path.relpath(split_dir, NECK_ROOT)   # e.g. "CT/train"
    out_dir = os.path.join(OUTPUT_ROOT, rel_dir)
    os.makedirs(out_dir, exist_ok=True)

    # Figure out which original csv (open/closed) each row came from, by
    # matching answer_type back to the filename convention used earlier.
    for answer_type, at_df in group_df.groupby("answer_type"):
        out_name = f"{answer_type.lower()}.csv"   # open.csv / closed.csv
        out_path = os.path.join(out_dir, out_name)
        at_df.drop(columns=["split_dir"]).to_csv(out_path, index=False)
        print(f"Saved: {out_path}  ({len(at_df)} rows)")

print(f"\nAll done. CoT-augmented train CSVs saved under: {OUTPUT_ROOT}")
print("Your original zip and extracted Abdomen folder were never modified.")


Saved: /kaggle/working/Neck_train_with_CoT/CT/train/closed.csv  (62 rows)
Saved: /kaggle/working/Neck_train_with_CoT/CT/train/open.csv  (67 rows)

All done. CoT-augmented train CSVs saved under: /kaggle/working/Neck_train_with_CoT
Your original zip and extracted Abdomen folder were never modified.


Let's convert the dataset into the "correct" format for finetuning:

The first example is now structured like below: